# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Norelle CHY–GLASSCOCK   Student ID: B00825108
2. Full name: Thien Anh Nguyen  Student ID: B00817290
3. Full name: Isabelle Liew Yu Chen  Student ID: B00819132
4. Full name: Lio SUCHET--WANG  Student ID: B00818171

**Group / repo name:** aidams-lab4-ChyGlasscock-LiewYuChen-Nguyen-SuchetWang

**Submitter (one person):** Lio SUCHETWANG

**Repo URL:** https://github.com/Lio0704/aidams-lab4-ChyGlasscock-LiewYuChen-Nguyen-SuchetWang.git

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).

## Setup (given)

In [3]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

excerpts = pd.DataFrame(json.loads(Path("excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads(Path("gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads(Path("recorded_responses.json").read_text()))
corpus = json.loads(Path("aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [4]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [5]:
ids = ["E03", "E09", "E19"]
for _, row in excerpts[excerpts["id"].isin(ids)].iterrows():
    print(f"===== {row['id']} =====")
    print(row["text"])
    print()

===== E03 =====
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

===== E09 =====
Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050.

===== E19 =====
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.



**Your answer (Q1):**

| field | E03 | E09 | E19 |
|---|---|---|---|
| company | Lianhe Iron & Steel Group | Ostend Metals NV | Bluewater Iron Corp. |
| report_year | 2023 | 2024 | 2024 |
| scope1_tco2 | 41,080,000 | null | 5,100,000 |
| scope2_tco2 | 5,120,000 | null | 410,000 |
| production_mt | 21.4 | 4.4 | 2.99 |
| net_zero_year | 2060 | 2050 | 2050 |

The hardest fields were the numeric ones that required interpretation rather than reading: scope1/scope2 in E03 (reported in units of 10,000 t), production_mt in E19 (short tons → metric), and scope1/scope2 in E09, where only a combined figure is given and the rule forces null. net_zero_year was also a judgment call in E03 (carbon neutrality 2060 vs. the 2030 peaking goal). If two annotators disagree on these, it shows the gold labels are themselves human judgments under an ambiguous spec, not objective truth. Part of the LLM’s measured “error” may therefore reflect label noise or rule ambiguity, so we should check inter-annotator agreement and treat small accuracy differences with caution.

## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [6]:
from pydantic import Field

class SteelDisclosure(BaseModel):
    company: str
    report_year: int
    scope1_tco2: Optional[float]
    scope2_tco2: Optional[float]
    production_mt: Optional[float]
    net_zero_year: Optional[int]
    confidence: float = Field(ge=0, le=1)


def parse_response(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None, "empty response"
    m = re.search(r"\{.*\}", raw, flags=re.S)
    if m is None:
        return None, "no JSON object found"
    try:
        record = SteelDisclosure.model_validate_json(m.group(0))
        return record, None
    except ValidationError as e:
        return None, str(e)


rows, failures = [], []
for r in recorded.to_dict("records"):
    record, err = parse_response(r["raw_text"])
    if record is not None:
        rows.append({"id": r["id"], "kind": r["kind"], "sample": r["sample"],
                     **record.model_dump()})
    else:
        failures.append({"id": r["id"], "kind": r["kind"], "sample": r["sample"],
                         "error": err, "raw_text": r["raw_text"]})

pred = pd.DataFrame(rows)
print(f"parsed: {len(pred)}   failed: {len(failures)}")
pred.head()

parsed: 119   failed: 1


,id,kind,sample,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year,confidence
0,E01,greedy,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
1,E01,sample,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.940
2,E01,sample,1,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
3,E01,sample,2,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.930
4,E01,sample,3,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.960


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [7]:
print("failed:", len(failures))
for f in failures:
    print(f"--- {f['id']} | {f['kind']} | sample {f['sample']}")
    print("error:", f["error"].splitlines()[0] if f["error"] else None)
    print("raw:  ", repr(str(f["raw_text"])[:300]))

def is_pure_json(raw):
    try:
        json.loads(raw.strip())
        return True
    except Exception:
        return False

ok_ids = set(zip(pred["id"], pred["kind"], pred["sample"]))
ok = recorded[[(i, k, s) in ok_ids for i, k, s in zip(recorded["id"], recorded["kind"], recorded["sample"])]]
needed_regex = ok[~ok["raw_text"].apply(is_pure_json)]
print("needed the regex:", len(needed_regex))

failed: 1
--- E11 | greedy | sample 0
error: 1 validation error for SteelDisclosure
raw:   '{\n  "company": "Anatolia Celik",\n  "report_year": 2024,\n  "scope1_tco2": 4120000,\n  "scope2_tco2": 2350000,\n  "production_mt": "8.1 Mt",\n  "net_zero_year": 2053,\n  "confidence": 0.94\n}'
needed the regex: 7


**Your answer (Q2):**

(a) Only 1 of 120 responses failed to parse: E11 (greedy). Its JSON is syntactically valid, but production_mt is the string "8.1 Mt" instead of a number, so pydantic rejects it (a string with a unit cannot be coerced to float). The value itself is probably right; the model just ignored the “number” type and the “convert all units” instruction for this field. 7 responses parsed only because the regex extracted the JSON from surrounding prose or code fences; a plain json.loads on the raw text would have failed for them.

(b) Option 1, automatic repair or retry: either clean the value (e.g. strip units like “Mt” and convert to a number) or re-send the prompt with the validation error appended. Risk: repair rules can silently mis-convert values (e.g. “8.1 Mt” vs “8.1 kt”, short tons vs metric tonnes), and a retry costs time and money and may return a different, possibly invented value, hiding the original problem.
Option 2, reject and flag for human review (store the record as failed, not as null). Risk: it needs human time and does not scale if failures become frequent. And if a failure were stored as null instead of flagged, downstream users could not tell “not reported” from “the model failed”.

Why count a failure as wrong on every field: in a real pipeline a response that does not parse delivers no usable data, so for the user every field is missing. If we dropped failures, we would only score the model on the responses it handled well, which inflates accuracy (survivorship bias) and could make a model that fails more often look better. Keeping all 120 responses in the denominator makes the accuracy figure reflect end-to-end usefulness and keeps it comparable across models and prompts.

> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [32]:
!pip install -q google-genai
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("ANTHROPIC_API_KEY"))

names = [m.name.replace("models/", "") for m in client.models.list()
         if "generateContent" in (m.supported_actions or [])]
candidates = [n for n in names if "flash" in n
              and not any(x in n for x in ["image", "tts", "live", "audio", "omni"])]
print("candidates:", candidates)

MODEL = None
for name in candidates:
    try:
        client.models.generate_content(model=name, contents="Say OK")
        MODEL = name
        break
    except Exception as e:
        print(f"  {name}: {str(e)[:80]}")
print("using model:", MODEL)

def ask_gemini(prompt):
    resp = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0))
    return resp.text

# 1. 在三条摘录上跑你的模型
ids = ["E03", "E09", "E19"]
texts = excerpts.set_index("id")["text"]

mine_rows = []
for eid in ids:
    raw = ask_gemini(PROMPT.replace("{text}", texts[eid]))
    rec, err = parse_response(raw)
    if rec is None:
        print(eid, "FAILED:", err.splitlines()[0])
        print(raw)
    else:
        mine_rows.append({"id": eid, **rec.model_dump()})
mine = pd.DataFrame(mine_rows).set_index("id")

# 2. 和 recorded greedy、gold 对比
fields6 = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
greedy = pred[pred["kind"] == "greedy"].set_index("id")

for eid in ids:
    cmp = pd.DataFrame({
        "my_model": mine.loc[eid, fields6] if eid in mine.index else None,
        "recorded_greedy": greedy.loc[eid, fields6],
        "gold": gold.loc[eid, fields6],
    })
    print(f"===== {eid} =====")
    display(cmp)

candidates: ['gemini-2.5-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash']
  gemini-2.5-flash: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
  gemini-flash-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
  gemini-flash-lite-latest: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
  gemini-2.5-flash-lite: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-
using model: gemini-3-flash-preview
===== E03 =====


,my_model,recorded_greedy,gold
company,Lianhe Iron & Steel Group,Lianhe Iron & Steel Group,Lianhe Iron & Steel Group
report_year,2023,2023,2023
scope1_tco2,"41,080,000.000","4,108,000.000","41,080,000.000"
scope2_tco2,"5,120,000.000","512,000.000","5,120,000.000"
production_mt,21.400,21.400,21.400
net_zero_year,2060,"2,060.000",2060


===== E09 =====


,my_model,recorded_greedy,gold
company,Ostend Metals NV,Ostend Metals,Ostend Metals
report_year,2024,2024,2024
scope1_tco2,NaN,"8,700,000.000",NaN
scope2_tco2,NaN,NaN,NaN
production_mt,4.400,4.400,4.400
net_zero_year,2050,"2,050.000",2050


===== E19 =====


,my_model,recorded_greedy,gold
company,Bluewater Iron Corp.,Bluewater Iron,Bluewater Iron
report_year,2024,2024,2024
scope1_tco2,"5,100,000.000","5,100,000.000","5,100,000.000"
scope2_tco2,"410,000.000","410,000.000","410,000.000"
production_mt,2.994,3.300,2.990
net_zero_year,2050,"2,050.000",2050


I ran Google AI Studio (Gemini, temperature 0) on E03, E09 and E19 and compared it with the recorded greedy answers and gold.

| Excerpt | Field | My model | Recorded greedy | Gold |
|---|---|---|---|---|
| E03 | scope1_tco2 | 41,080,000 ✓ | 4,108,000 ✗ | 41,080,000 |
| E03 | scope2_tco2 | 5,120,000 ✓ | 512,000 ✗ | 5,120,000 |
| E09 | scope1_tco2 | null ✓ | 8,700,000 ✗ | null |
| E19 | production_mt | 2.994 ✓ | 3.300 ✗ | 2.990 |

All other fields agree across the three columns. My model got all 18 fields right, while the recorded greedy model made 4 errors on the same excerpts. It did not make the same mistakes: it converted "10,000 tonnes" correctly, refused to split the combined Scope 1+2 figure, and converted short tons to metric tonnes (2.994 Mt, within the 2% tolerance).

This is exactly the "second model" fix from Q7: E03 scope1 and E19 production were the two errors where all five recorded samples agreed, so resampling the recorded model could never reveal them, but a model with a different structure did. Caveats: this is only 3 excerpts, and they were chosen as traps, so it says nothing reliable about the new model's overall accuracy; it might fail on other cases (e.g. invented values or table years). A proper comparison would rerun all 20 excerpts and apply the same scoring and intervals as in Part B.

---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [12]:
FIELDS6 = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
YEAR_FIELDS = {"report_year", "net_zero_year"}
NUM_FIELDS = {"scope1_tco2", "scope2_tco2", "production_mt"}

def _norm(s):
    s = re.sub(r"[^a-z0-9]+", " ", str(s).lower())
    return " ".join(s.split())

def field_correct(field, pred_value, gold_value):
    p_missing, g_missing = pd.isna(pred_value), pd.isna(gold_value)
    if p_missing and g_missing:
        return True
    if p_missing or g_missing:
        return False
    if field == "company":
        p, g = _norm(pred_value), _norm(gold_value)
        return p in g or g in p
    try:
        p, g = float(pred_value), float(gold_value)
    except (TypeError, ValueError):
        return False
    if field in YEAR_FIELDS:
        return int(p) == int(g)
    if field in NUM_FIELDS:
        if g == 0:
            return p == 0
        return abs(p - g) <= 0.02 * abs(g)
    return pred_value == gold_value

def score(df):
    df = df.set_index("id") if "id" in df.columns else df
    parsed_ids = set(df.index)
    df = df.reindex(gold.index)
    rows = []
    for eid in gold.index:
        for f in FIELDS6:
            p, g = df.loc[eid, f], gold.loc[eid, f]
            ok = field_correct(f, p, g) if eid in parsed_ids else False
            rows.append({"id": eid, "field": f, "pred": p, "gold": g, "correct": ok})
    return pd.DataFrame(rows)

greedy_pred = pred[pred["kind"] == "greedy"]
greedy_scores = score(greedy_pred)

acc_by_field = (greedy_scores.groupby("field")["correct"].mean()
                .reindex(FIELDS6).to_frame("accuracy"))
acc_by_field

,accuracy
field,
company,0.950
report_year,0.900
scope1_tco2,0.750
scope2_tco2,0.850
production_mt,0.900
net_zero_year,0.850


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [11]:
print(f"overall accuracy: {greedy_scores['correct'].mean():.3f}")
print("worst field:", acc_by_field["accuracy"].idxmin(),
      f"({acc_by_field['accuracy'].min():.3f})")

texts = excerpts.set_index("id")["text"]
wrong = greedy_scores[~greedy_scores["correct"]]
print(f"\n{len(wrong)} wrong (excerpt, field) pairs\n")
for eid, g in wrong.groupby("id"):
    print(f"===== {eid} =====")
    print(texts[eid])
    display(g[["field", "pred", "gold"]])

overall accuracy: 0.867
worst field: scope1_tco2 (0.750)

16 wrong (excerpt, field) pairs

===== E03 =====
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.


,field,pred,gold
14,scope1_tco2,"4,108,000.000","41,080,000.000"
15,scope2_tco2,"512,000.000","5,120,000.000"


===== E05 =====
Tamarind Steel Ltd - Business Responsibility and Sustainability Report FY2024. Crude steel production rose 6% to 18.2 million tonnes. Our CO2 emission intensity improved to 2.31 tCO2 per tonne of crude steel (tcs), from 2.38 tCO2/tcs a year earlier. Tamarind Steel targets net zero by 2070, in line with India's national commitment.


,field,pred,gold
26,scope1_tco2,"42,042,000.000",NaN


===== E07 =====
Great Lakes Ironworks Inc. - 2024 Sustainability Report. Key performance data: Scope 1 emissions (MtCO2e) 2022: 14.9 | 2023: 14.2 | 2024: 13.8. Scope 2 emissions (MtCO2e) 2022: 1.1 | 2023: 1.0 | 2024: 0.96. Raw steel production (Mt) 2022: 7.9 | 2023: 7.6 | 2024: 7.4. We remain committed to net zero by 2050.


...

,field,pred,gold
105,scope2_tco2,0.000,NaN


===== E19 =====
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.


,field,pred,gold
112,production_mt,3.300,2.990


===== E20 =====
Nordic Green Steel AB - Annual Report 2024. Construction of our hydrogen-based plant is on track; first coil is expected in 2026 and the plant is designed for 2.5 Mt per year of near-zero emission steel. Scope 1 emissions in 2024, mainly from temporary gas boilers on site: 12,400 tCO2e. Scope 2: 3,900 tCO2e.


,field,pred,gold
119,net_zero_year,"2,026.000",<NA>


**Your answer (Q3):**

**(a)** Overall greedy accuracy is **0.867** (104 of 120 field values correct). The worst field is **scope1_tco2 (0.75)**, followed by scope2_tco2 and net_zero_year (0.85 each).

**(b)** The 16 errors fall into five types:

| Error type | Cases | Count |
|---|---|---|
| Parse failure | E11 (all 6 fields) | 6 |
| Value invented or computed for a missing field | E05 scope1 (intensity × production), E18 scope2 (0 for "not yet disclosed"), E10 net_zero_year (2030 intensity target), E20 net_zero_year (2026 plant start-up) | 4 |
| Unit conversion | E03 scope1 & scope2 ("10,000 tonnes" treated as thousands), E19 production_mt (short tons not converted) | 3 |
| Wrong year | E07 scope1 (2023 column instead of 2024), E08 report_year (publication year 2025 instead of FY2024 → 2024) | 2 |
| Wrong scope | E09 scope1 (combined Scope 1+2 put into Scope 1) | 1 |

Counted by field, parse failure looks largest, but it is a single response. Counted by distinct mistakes, the most frequent type is inventing or computing a value that is not reported (4 excerpts), which directly violates the prompt rule "Never compute a value that is not reported". The model tends to fill a slot rather than return null. Notably, E11 would still have been wrong even if it had parsed: the model reported shipments (8.1 Mt) instead of liquid steel production (9.8 Mt).

**(c)** The model fell into all three traps from my Q1 excerpts (E03 unit, E09 combined scope, E19 short tons). These are errors I could also have made: unit conversions and picking the right year or column in a table or fiscal-year statement require careful reading and are easy for a human to slip on. I would have been much less likely to invent values, e.g. compute Scope 1 from an intensity figure, write 0 for "will be disclosed", or read a plant start-up date as a net zero target. A human annotator tends to leave a field blank when unsure, while the model prefers to produce a plausible number.

## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [13]:
# 1. Bayesian: Beta-Binomial
k = int(greedy_scores["correct"].sum())
n = len(greedy_scores)
beta_ci = stats.beta(1 + k, 1 + n - k).ppf([0.05, 0.95])

# 2. Bootstrap over excerpts
per = greedy_scores.groupby("id")["correct"].agg(["sum", "count"])
sums, counts = per["sum"].to_numpy(), per["count"].to_numpy()

rng = np.random.default_rng(SEED)
B, m = 2000, len(per)
idx = rng.integers(0, m, size=(B, m))          # 每行是一次有放回抽取的 m 个摘录
boot_acc = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
boot_ci = np.percentile(boot_acc, [5, 95])

print(f"k = {k}, n = {n}, accuracy = {k/n:.3f}")
print(f"Beta 90% credible interval:  [{beta_ci[0]:.3f}, {beta_ci[1]:.3f}]")
print(f"Bootstrap 90% interval:      [{boot_ci[0]:.3f}, {boot_ci[1]:.3f}]")

k = 104, n = 120, accuracy = 0.867
Beta 90% credible interval:  [0.806, 0.908]
Bootstrap 90% interval:      [0.775, 0.933]


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

**(a)** Greedy accuracy is 104/120 = **0.867**.

| Method | 90% interval | Width |
|---|---|---|
| Beta-Binomial, Beta(105, 17) | [0.806, 0.908] | 0.10 |
| Bootstrap over 20 excerpts (2,000 resamples) | [0.775, 0.933] | 0.16 |

The **bootstrap interval is wider**. The Beta-Binomial treats the 120 fields as 120 independent coin flips, but the six fields of one excerpt are not independent: errors cluster within excerpts. E11 is the extreme case, where one parse failure made all six fields wrong at once, and E03 lost two fields to the same unit mistake. The real unit of sampling is the excerpt, so we effectively have 20 observations, not 120. Resampling excerpts keeps these clusters together: whether a resample contains E11 zero, one or two times moves accuracy by about 5 points, which widens the interval. The Beta interval is therefore overconfident.

**(b)** Interval width shrinks roughly with 1/√n, so halving the width needs about **4× as many excerpts**: around 80 excerpts (≈480 fields) instead of 20, i.e. about 60 more to label.

**(c)** No, not really. 95% lies above the upper end of both intervals (0.908 for Beta, 0.933 for the bootstrap). In the bootstrap, fewer than 3% of resamples reach 95%. On our data the model looks more like an 80–90% extractor. Caveats: the vendor may measure on a different, easier dataset, with a different model, looser matching rules, or accuracy over parsed responses only, and our sample of 20 excerpts is small. But on our task, our gold labels and our scoring, the evidence does not support a 95% claim. A vendor figure should be checked on our own labeled sample before we rely on it.

In [14]:
print(f"share of bootstrap resamples with accuracy ≥ 0.95: {(boot_acc >= 0.95).mean():.3f}")

share of bootstrap resamples with accuracy ≥ 0.95: 0.022


## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [15]:
g = greedy_pred.set_index("id").reindex(gold.index)

downstream = pd.DataFrame({
    "scope1_llm":  pd.to_numeric(g["scope1_tco2"], errors="coerce"),
    "scope1_gold": pd.to_numeric(gold["scope1_tco2"], errors="coerce"),
}, index=gold.index)
downstream["error_t"] = downstream["scope1_llm"].fillna(0) - downstream["scope1_gold"].fillna(0)

total_llm  = downstream["scope1_llm"].fillna(0).sum() / 1e6
total_gold = downstream["scope1_gold"].fillna(0).sum() / 1e6
net_err    = downstream["error_t"].sum() / 1e6
abs_err    = downstream["error_t"].abs().sum() / 1e6

int_llm  = (downstream["scope1_llm"]  / (pd.to_numeric(g["production_mt"], errors="coerce") * 1e6)).dropna()
int_gold = (downstream["scope1_gold"] / (pd.to_numeric(gold["production_mt"], errors="coerce") * 1e6)).dropna()

print(f"Total Scope 1  LLM:  {total_llm:8.2f} MtCO2")
print(f"Total Scope 1  gold: {total_gold:8.2f} MtCO2")
print(f"Net error:           {net_err:+8.2f} MtCO2")
print(f"Absolute error:      {abs_err:8.2f} MtCO2")
print(f"Median intensity LLM:  {int_llm.median():.3f} tCO2/t  (n = {len(int_llm)})")
print(f"Median intensity gold: {int_gold.median():.3f} tCO2/t  (n = {len(int_gold)})")

downstream[downstream["error_t"] != 0]

Total Scope 1  LLM:    252.11 MtCO2
Total Scope 1  gold:   242.06 MtCO2
Net error:             +10.05 MtCO2
Absolute error:         92.23 MtCO2
Median intensity LLM:  1.895 tCO2/t  (n = 18)
Median intensity gold: 1.865 tCO2/t  (n = 17)


,scope1_llm,scope1_gold,error_t
id,,,
E03,"4,108,000.000","41,080,000.000","-36,972,000.000"
E05,"42,042,000.000",NaN,"42,042,000.000"
E07,"14,200,000.000","13,800,000.000","400,000.000"
E09,"8,700,000.000",NaN,"8,700,000.000"
E11,NaN,"4,120,000.000","-4,120,000.000"


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

**(a)** The LLM gives a sector total of **252.11 MtCO2** against **242.06 MtCO2** in gold.

| | MtCO2 | % of gold total |
|---|---|---|
| Net error | +10.05 | +4.2% |
| Absolute error | 92.23 | 38.1% |

The two largest contributors are **E05 (+42.04 Mt)**, where Scope 1 was computed from intensity × production although it is not reported, and **E03 (−36.97 Mt)**, where "10,000 tonnes" was converted as thousands. Together they account for 79.0 Mt, about **86% of the absolute error**. The net error looks small only because these two large errors have opposite signs and almost cancel; that is luck, not accuracy.

**(b)** Field accuracy counts every field equally: a wrong Scope 1 for a 41 Mt emitter weighs the same as a wrong company name. A sector total instead weights each error by its size in tonnes, and the error sizes are heavy-tailed: a unit mistake is off by a factor of 10, an invented value adds a whole company's emissions. So 5 wrong Scope 1 values out of 20 (75% accuracy on that field, 87% overall) are enough to move the total by tens of Mt. This is the Session 4 lesson: the mean (accuracy, or a sum) hides the tail, and the damage lives in a few extreme cases.

The median intensity barely moves (1.895 vs 1.865 tCO2/t, +1.6%) because the median depends only on the middle of the ranking. A few wrong companies shift position, e.g. E03 drops from about 1.92 to 0.19 tCO2/t, but that moves the median by at most one or two places. The median is robust to the same outliers that dominate the total. Note also that the two medians are based on different sets (LLM n = 18, gold n = 17), because the LLM invented values for E05 and E09 and lost E11.

**(c)** Alongside the total I would report:

- the **absolute error and an uncertainty range** (e.g. a bootstrap interval over companies), not just the point estimate or the net error;
- the **per-company contributions**, flagging the largest emitters, which drive the total and should be checked by hand;
- the **coverage**: how many companies were parse failures or had no reported Scope 1 (here E11 failed; E05 and E09 do not report Scope 1 separately) and how missing values were treated (counted as 0 here, which biases the total downward);
- which values were **taken directly from the report vs inferred**, so invented numbers like E05 are not mixed into the total unnoticed.

---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [17]:
samples = pred[pred["kind"] == "sample"]

def _key(v):
    if pd.isna(v):
        return json.dumps(None)
    if hasattr(v, "item"):
        v = v.item()
    return json.dumps(v)

rows = []
for eid in gold.index:
    s = samples[samples["id"] == eid]
    for f in FIELDS6:
        keys = [_key(v) for v in s[f]]
        if keys:
            counts = pd.Series(keys).value_counts()
            vote, n_agree = json.loads(counts.index[0]), int(counts.iloc[0])
            ok = field_correct(f, vote, gold.loc[eid, f])
        else:
            vote, n_agree, ok = None, 0, False
        rows.append({"id": eid, "field": f, "vote": vote,
                     "agreement": n_agree / 5, "correct": ok})

votes = pd.DataFrame(rows)

print(f"greedy accuracy:        {greedy_scores['correct'].mean():.3f}")
print(f"majority-vote accuracy: {votes['correct'].mean():.3f}")

n5 = (votes["agreement"] * 5).round().astype(int)
votes["agree_group"] = np.select([n5 == 5, n5 == 4], ["5/5", "4/5"], default="≤3/5")
(votes.groupby("agree_group")["correct"]
      .agg(accuracy="mean", n="count")
      .reindex(["5/5", "4/5", "≤3/5"]))

greedy accuracy:        0.867
majority-vote accuracy: 0.950


,accuracy,n
agree_group,,
5/5,0.978,93
4/5,0.933,15
≤3/5,0.750,12


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [18]:
bins = (votes.groupby("agree_group")["correct"]
              .agg(accuracy="mean", n="count", k="sum")
              .reindex(["5/5", "4/5", "≤3/5"]))
print(f"greedy accuracy: {greedy_scores['correct'].mean():.3f}")
print(f"vote accuracy:   {votes['correct'].mean():.3f}")
display(bins)

k5 = int(bins.loc["5/5", "k"])
n5 = int(bins.loc["5/5", "n"])
post = stats.beta(1 + k5, 1 + n5 - k5)
ci5 = post.ppf([0.05, 0.95])
print(f"5/5: {k5}/{n5} = {k5/n5:.3f},  90% interval [{ci5[0]:.3f}, {ci5[1]:.3f}]")
print(f"P(accuracy > 0.95 | data) = {1 - post.cdf(0.95):.3f}")

greedy accuracy: 0.867
vote accuracy:   0.950


,accuracy,n,k
agree_group,,,
5/5,0.978,93,91
4/5,0.933,15,14
≤3/5,0.750,12,9


5/5: 91/93 = 0.978,  90% interval [0.935, 0.991]
P(accuracy > 0.95 | data) = 0.855


**Your answer (Q6):**

**(a)** Majority voting over the 5 samples raises accuracy from **0.867 (greedy)** to **0.950 (vote)**, i.e. from 16 wrong fields to 6.

| Agreement | Accuracy | Fields | Wrong |
|---|---|---|---|
| 5/5 | 0.978 | 93 | 2 |
| 4/5 | 0.933 | 15 | 1 |
| ≤3/5 | 0.750 | 12 | 3 |

**(b)** **Yes, but it is not a complete warning.** Accuracy falls steadily with agreement: the error rate is about 2% at 5/5 but 25% at ≤3/5, more than ten times higher. Flagging only the ≤3/5 fields would send 12 of 120 fields (10%) to a human and catch half of the remaining errors (3 of 6). So disagreement is a cheap and useful triage signal. But it misses errors where the model is **consistently wrong**: 2 errors have full 5/5 agreement. These are systematic mistakes (e.g. the same misreading of a unit or rule every time), and sampling cannot reveal them, just as an ensemble of similar models shares the same blind spots. High agreement means the model is consistent, not that it is correct.

**(c)** For the 5/5 bin, 91 of 93 fields are correct (0.978). With a Beta(1, 1) prior the posterior is Beta(92, 3), with a **90% credible interval of [0.935, 0.991]**. The lower bound is below 0.95, so **we cannot claim accuracy above 95%** at this level of confidence (the posterior probability that it exceeds 95% is about 0.855). And this interval is still optimistic: like in Q4, it treats the 93 fields as independent, while fields of the same excerpt are correlated, so the true uncertainty is larger. With only 93 fields, two errors are enough to make the claim unproven.

## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [19]:
confident_wrong = votes[(votes["agreement"] == 1.0) & (~votes["correct"])].copy()
confident_wrong["gold"] = [gold.loc[i, f] for i, f in zip(confident_wrong["id"], confident_wrong["field"])]

print(f"{len(confident_wrong)} fields where all 5 samples agree and the vote is wrong\n")
for eid, g in confident_wrong.groupby("id"):
    print(f"===== {eid} =====")
    print(texts[eid])
    display(g[["field", "vote", "gold", "agreement"]])

2 fields where all 5 samples agree and the vote is wrong

===== E03 =====
Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.


,field,vote,gold,agreement
14,scope1_tco2,"4,108,000.000","41,080,000.000",1.000


===== E19 =====
Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.


,field,vote,gold,agreement
112,production_mt,3.300,2.990,1.000


**Your answer (Q7):**

Two fields are wrong even though all five samples agree:

| Excerpt | Field | Vote | Gold | Error |
|---|---|---|---|---|
| E03 | scope1_tco2 | 4,108,000 | 41,080,000 | "unit: 10,000 tonnes" read as thousands, off by a factor of 10 |
| E19 | production_mt | 3.30 | 2.99 | 3.3 million **short** tons not converted to metric tonnes, about +10% |

**(a)** Both are **unit-conversion errors on unusual units**: a Chinese-style "10,000 tonnes" unit (万吨) and US short tons. In both cases the model copies the number correctly but attaches a plausible, wrong unit, so the output looks perfectly reasonable. The error is not random: the model misreads the unit in the same way every time.

**(b)** Sampling at temperature 0.7 only shows the variability **inside one model**, the analogue of aleatoric noise around the model's answer. These two errors are **epistemic, and more precisely structural (model) uncertainty**: the model's understanding of these units is wrong, and every sample inherits the same flaw. The five samples are not independent draws; their errors are perfectly correlated, so agreement 5/5 measures consistency, not correctness. More samples from the same model cannot shrink an uncertainty that lies in the model itself.

Which fix would reveal them:

- **A unit-conversion rule** is the most reliable: ask the model to return the raw number and the unit as written, then convert in code with an explicit table (10,000 t, kt, Mt, short tons…). This removes the structural error at its source.
- **A second model** can reveal it if its structure differs: disagreement between models is itself information about structural uncertainty. But models trained on similar data may share the same blind spot, so agreement between them is not proof.
- **A human** who knows these units would catch both, but only if the field is reviewed; agreement 5/5 would not flag it.
- **A different prompt** that names these units may fix these two cases, but only for the units we anticipated.

**(c)** The analogue is the linear-trend extrapolation of the loss index in the Session 4 worked example. The true process grew exponentially (15%/yr); the linear model's 95% band looked tight and confident, yet 3 of the 5 outcomes in 2025–2029 fell outside it. The interval only quantified uncertainty **inside the chosen model** and could not see its structural error, just as five samples from the same LLM cannot see its unit misreading. The fix was the same too: compare a **different model** (Model B, exponential), whose disagreement with Model A revealed the structural uncertainty. And as with CMIP6, an ensemble only helps if its members do not share the same flaws.

## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [20]:
samp = pred[pred["kind"] == "sample"].copy()

samp["all_correct"] = [
    all(field_correct(f, r[f], gold.loc[r["id"], f]) for f in FIELDS6)
    for _, r in samp.iterrows()
]

print(f"responses:              {len(samp)}")
print(f"mean stated confidence: {samp['confidence'].mean():.3f}")
print(f"share fully correct:    {samp['all_correct'].mean():.3f}")

samp["conf_bin"] = pd.cut(samp["confidence"], [0, .90, .93, .96, 1])
calib = (samp.groupby("conf_bin", observed=False)
             .agg(mean_conf=("confidence", "mean"),
                  share_correct=("all_correct", "mean"),
                  n=("all_correct", "size")))
display(calib)

c_ok  = samp.loc[samp["all_correct"], "confidence"]
c_bad = samp.loc[~samp["all_correct"], "confidence"]
if len(c_ok) and len(c_bad):
    U = stats.mannwhitneyu(c_ok, c_bad).statistic
    print(f"AUC (confidence ranks correct above wrong): {U / (len(c_ok) * len(c_bad)):.3f}")

responses:              100
mean stated confidence: 0.928
share fully correct:    0.550


,mean_conf,share_correct,n
conf_bin,,,
"(0.0, 0.9]",0.890,0.286,21
"(0.9, 0.93]",0.921,0.457,35
"(0.93, 0.96]",0.947,0.737,38
"(0.96, 1.0]",0.978,0.833,6


AUC (confidence ranks correct above wrong): 0.732


**Your answer (Q8):**

Over the 100 sample responses, the mean stated confidence is **0.928**, but only **55%** of responses have all six fields correct. PROMPT defines confidence as "how confident you are that every field is correct", so this is exactly the right comparison.

| Stated confidence | Mean confidence | Share fully correct | n |
|---|---|---|---|
| ≤ 0.90 | 0.890 | 0.286 | 21 |
| 0.90–0.93 | 0.921 | 0.457 | 35 |
| 0.93–0.96 | 0.947 | 0.737 | 38 |
| > 0.96 | 0.978 | 0.833 | 6 |

**Is the model over-confident? Yes, strongly.** On average it overstates its chance of being fully right by about 38 percentage points, and it does so in every bin: responses that say 0.89 are fully correct only 29% of the time. The stated numbers are squeezed into a narrow band (roughly 0.85 to 1), so they cannot be read as probabilities. As in Session 4, "a 95% label is a claim, test it": back-tested against gold, this claim fails badly.

**Is it still useful for ranking? Yes, moderately.** The share fully correct rises steadily with stated confidence (29% → 46% → 74% → 83%), and the AUC is **0.73**: a randomly chosen fully correct response gets a higher confidence than a randomly chosen wrong one about 73% of the time (0.5 would mean no information). So confidence carries real **relative** information even though its **absolute** level is wrong. In practice we could use it to rank responses and send the lowest-confidence ones to human review, or recalibrate it on a labeled set (map stated confidence to observed accuracy) before reporting it as a probability. Caveats: the top bin has only 6 responses, and the 5 samples per excerpt are not independent, so the bin estimates are noisy. And as Q7 showed, confidence will not catch errors the model is systematically sure about.

## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [21]:
COST_REVIEW, COST_ERROR = 4, 150
n_agree = (votes["agreement"] * 5).round().astype(int)

policies = {
    "P0: trust vote":      pd.Series(False, index=votes.index),
    "P1: review < 5/5":    n_agree < 5,
    "P2: review <= 3/5":   n_agree <= 3,
}

rows = []
for name, review in policies.items():
    n_review = int(review.sum())
    remaining = int((~votes["correct"] & ~review).sum())
    rows.append({"policy": name, "reviewed": n_review, "errors_left": remaining,
                 "review_cost": n_review * COST_REVIEW,
                 "error_cost": remaining * COST_ERROR,
                 "total_cost": n_review * COST_REVIEW + remaining * COST_ERROR})
cost_table = pd.DataFrame(rows).set_index("policy")
display(cost_table)

best = cost_table["total_cost"].idxmin()
survivors = votes[~votes["correct"] & ~policies[best]]
print("cheapest:", best)
display(survivors[["id", "field", "vote", "agreement"]])

,reviewed,errors_left,review_cost,error_cost,total_cost
policy,,,,,
P0: trust vote,0,6,0,900,900
P1: review < 5/5,27,2,108,300,408
P2: review <= 3/5,12,3,48,450,498


cheapest: P1: review < 5/5


,id,field,vote,agreement
14,E03,scope1_tco2,"4,108,000.000",1.000
112,E19,production_mt,3.300,1.000


**Your answer (Q9):**

| Policy | Fields reviewed | Errors left | Review cost | Error cost | Total cost (EUR) |
|---|---|---|---|---|---|
| P0: trust the vote | 0 | 6 | 0 | 900 | **900** |
| P1: review agreement < 5/5 | 27 | 2 | 108 | 300 | **408** |
| P2: review agreement ≤ 3/5 | 12 | 3 | 48 | 450 | **498** |

**P1 is the cheapest** (EUR 408), less than half the cost of trusting the model (EUR 900). Going from P2 to P1 means reviewing 15 more fields (EUR 60) to catch one more error (EUR 150 saved), which is worth it.

A simple rule explains this: reviewing a field pays off when its error probability exceeds the break-even 4 / 150 ≈ **2.7%**. The 4/5 bin has an error rate of 1/15 ≈ 6.7%, above the threshold, so it should be reviewed; the 5/5 bin has 2/93 ≈ 2.2%, just below it, so reviewing all 93 fields (EUR 372) would cost more than the two errors it catches (EUR 300). The decision depends on these two prices: if an error cost more than about EUR 190, reviewing everything would become worthwhile.

**Errors that survive P1:** the two 5/5 errors from Q7, **E03 scope1_tco2** ("10,000 tonnes" read as thousands) and **E19 production_mt** (short tons not converted). Agreement cannot flag them because the model is consistently wrong.

**One extra check that would catch them: a unit flag.** Send to a human any field whose excerpt contains a non-standard unit ("10,000 tonnes", "short tons", "kt", 万吨…), or better, have the model return the raw number and unit and convert in code. This targets exactly the structural error that resampling cannot see, at the cost of only a few extra reviews. A plausibility check on intensity (Scope 1 / production) would also catch E03, whose implied intensity of 0.19 tCO2/t is ten times below the sector median, but it would miss E19, where the error is only about 10%.

---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks_raw = corpus["chunks"]
chunks = [c if isinstance(c, str) else (c.get("text") or c.get("content")) for c in chunks_raw]

def pick(d, names):
    for n in names:
        if n in d:
            return d[n]
    raise KeyError(f"none of {names} in {list(d.keys())}")

queries = corpus["queries"]
q_texts = [pick(q, ["question", "query", "text", "q"]) for q in queries]
q_gold = [pick(q, ["relevant", "chunk", "answer_chunk", "chunk_id", "relevant_chunk", "gold", "idx"])
          for q in queries]
q_gold = [[int(x) for x in (g if isinstance(g, list) else [g])] for g in q_gold]   # 统一成整数列表

vec = TfidfVectorizer(stop_words="english")
X_chunks = vec.fit_transform(chunks)
X_q = vec.transform(q_texts)
S = cosine_similarity(X_q, X_chunks)

rows = []
for i, (qt, g) in enumerate(zip(q_texts, q_gold)):
    top3 = np.argsort(-S[i])[:3]
    rows.append({"query": qt, "relevant": g, "top3": top3.tolist(),
                 "top3_scores": np.round(S[i, top3], 3).tolist(),
                 "hit@1": int(top3[0]) in g,
                 "hit@3": any(int(t) in g for t in top3)})
retrieval = pd.DataFrame(rows)

print(f"hit@1 = {retrieval['hit@1'].mean():.2f}   hit@3 = {retrieval['hit@3'].mean():.2f}")
retrieval

hit@1 = 0.67   hit@3 = 0.67


,query,relevant,top3,top3_scores,hit@1,hit@3
0,What were Scope 1 emissions in 2024?,[3],"[1, 5, 12]","[0.338, 0.273, 0.241]",False,False
1,net zero target year,[5],"[5, 0, 13]","[0.366, 0.133, 0.097]",True,True
2,crude steel production 2024,[2],"[2, 3, 0]","[0.347, 0.327, 0.209]",True,True
3,investment in hydrogen DRI capex,[6],"[6, 0, 2]","[0.225, 0.0, 0.0]",True,True
4,impact of the carbon price on EBITDA,[8],"[8, 13, 7]","[0.346, 0.237, 0.22]",True,True
5,flooding exposure of our plants,[9],"[3, 6, 1]","[0.208, 0.196, 0.0]",False,False


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [24]:
print(f"hit@1 = {retrieval['hit@1'].mean():.2f} ({retrieval['hit@1'].sum()}/{len(retrieval)})")
print(f"hit@3 = {retrieval['hit@3'].mean():.2f} ({retrieval['hit@3'].sum()}/{len(retrieval)})\n")

for i, r in retrieval[~retrieval["hit@1"]].iterrows():
    first = r["top3"][0]
    print("=" * 80)
    print("QUERY:", r["query"])
    print("query terms used by TF-IDF:", list(vec.inverse_transform(X_q[i])[0]))
    print(f"\n--- retrieved first: chunk {first} (score {S[i, first]:.3f}) ---")
    print(chunks[first])
    for g in r["relevant"]:
        print(f"\n--- relevant: chunk {g} (score {S[i, g]:.3f}) ---")
        print(chunks[g])
    print()

hit@1 = 0.67 (4/6)
hit@3 = 0.67 (4/6)

QUERY: What were Scope 1 emissions in 2024?
query terms used by TF-IDF: [np.str_('2024'), np.str_('emissions'), np.str_('scope')]

--- retrieved first: chunk 1 (score 0.338) ---
About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 covers purchased electricity and heat, and Scope 3 covers our value chain.

--- relevant: chunk 3 (score 0.184) ---
Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.

QUERY: flooding exposure of our plants
query terms used by TF-IDF: [np.str_('plants')]

--- retrieved first: chunk 3 (score 0.208) ---
Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.

--- relevant:

**Your answer (Q10):**

**(a)** TF-IDF retrieval gives **hit@1 = 0.67 and hit@3 = 0.67** (4 of 6 queries). Both failures are complete misses: the relevant chunk is not even in the top 3.

**(b)** Two different failure modes:

| Query | Retrieved first | Relevant chunk | Why TF-IDF failed |
|---|---|---|---|
| What were Scope 1 emissions in 2024? | chunk 1 (0.338): definitions of Scope 1/2/3 | chunk 3 (0.184): "Direct greenhouse gas emissions… 24.19 MtCO2e in 2024" | **Synonym + distractor** |
| flooding exposure of our plants | chunk 3 (0.208): "…sinter plants…" | chunk 9 (0.000): "riverine inundation… most exposed" | **Vocabulary mismatch + word sense** |

*Query 1:* the answer chunk never says "Scope 1"; it says "direct emissions", which means the same thing. The tokenizer even drops the "1" (single-character tokens are ignored), so the query becomes just {2024, emissions, scope}. Chunk 1, a methodology paragraph that defines the scopes, repeats "Scope" and "emissions" several times and wins on word overlap, although it contains no number at all.

*Query 2:* only one query word survives, "plants". "Flooding" and "exposure" never appear in the chunks, while the relevant chunk uses "inundation" and "exposed". TF-IDF does no stemming and knows no synonyms, so chunk 9 scores exactly 0. The one shared word is then matched in the wrong sense: "sinter plants" (equipment) in chunk 3 instead of "our plants" (= sites).

**(c)** Embeddings would most likely fix **the flooding query**: "flooding" ≈ "riverine inundation", "exposure" ≈ "most exposed", "plants" ≈ "sites" are close in meaning even with no word in common, and chunk 9 is the only chunk about physical risk. They would probably also help the Scope 1 query, since "direct emissions" and "Scope 1 emissions" mean the same thing, but this case is less certain: the definitions chunk is *also* about Scope 1 in meaning, so a dense retriever may still rank it high. A query asking for a value, or a hybrid retriever (keywords + embeddings), would be more robust there.

**(d)** An extraction error happens on the right text: the evidence is in front of the model (and of a reviewer), so a wrong number can be checked against the source, and signals like disagreement between samples can flag it. A retrieval miss removes the evidence before the LLM ever sees it. The model then either says "not reported", a false negative that looks like a legitimate null, or, worse, answers confidently from the wrong chunk (e.g. a definition or a target instead of the reported value). Both look like normal, well-formed outputs: the citation points to a real chunk, the samples may all agree, and extraction accuracy measured on the retrieved text can look perfect. The error is **upstream and invisible downstream**, much like the structural errors of Q7: resampling or voting cannot fix what was never retrieved. That is why retrieval has to be evaluated separately (hit@k), and why critical fields should be checked against the full document.

> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [33]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("all-MiniLM-L6-v2")
E_chunks = emb_model.encode(chunks, normalize_embeddings=True)
E_q = emb_model.encode(q_texts, normalize_embeddings=True)
S_dense = cosine_similarity(E_q, E_chunks)

rows = []
for i, (qt, g) in enumerate(zip(q_texts, q_gold)):
    top3 = np.argsort(-S_dense[i])[:3]
    rows.append({"query": qt, "relevant": g, "top3": top3.tolist(),
                 "top3_scores": np.round(S_dense[i, top3], 3).tolist(),
                 "hit@1": int(top3[0]) in g,
                 "hit@3": any(int(t) in g for t in top3)})
retrieval_dense = pd.DataFrame(rows)

print(f"TF-IDF : hit@1 = {retrieval['hit@1'].mean():.2f}   hit@3 = {retrieval['hit@3'].mean():.2f}")
print(f"Dense  : hit@1 = {retrieval_dense['hit@1'].mean():.2f}   hit@3 = {retrieval_dense['hit@3'].mean():.2f}")
display(pd.DataFrame({
    "query": q_texts,
    "TF-IDF top1": retrieval["top3"].str[0],
    "Dense top1": retrieval_dense["top3"].str[0],
    "relevant": q_gold,
    "TF-IDF hit@1": retrieval["hit@1"],
    "Dense hit@1": retrieval_dense["hit@1"],
}))

for i, r in retrieval_dense[~retrieval_dense["hit@1"]].iterrows():
    first = r["top3"][0]
    print("=" * 80)
    print("STILL FAILS:", r["query"])
    print(f"--- retrieved first: chunk {first} ({S_dense[i, first]:.3f}) ---\n{chunks[first]}")
    for g in r["relevant"]:
        print(f"--- relevant: chunk {g} ({S_dense[i, g]:.3f}) ---\n{chunks[g]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TF-IDF : hit@1 = 0.67   hit@3 = 0.67
Dense  : hit@1 = 0.83   hit@3 = 0.83


,query,TF-IDF top1,Dense top1,relevant,TF-IDF hit@1,Dense hit@1
0,What were Scope 1 emissions in 2024?,1,5,[3],False,False
1,net zero target year,5,5,[5],True,True
2,crude steel production 2024,2,2,[2],True,True
3,investment in hydrogen DRI capex,6,6,[6],True,True
4,impact of the carbon price on EBITDA,8,8,[8],True,True
5,flooding exposure of our plants,3,9,[9],False,True


STILL FAILS: What were Scope 1 emissions in 2024?
--- retrieved first: chunk 5 (0.577) ---
Targets. We are committed to reaching net zero greenhouse gas emissions by 2050, with an interim target to reduce Scope 1 and 2 emissions by 35% by 2030 against a 2018 baseline.
--- relevant: chunk 3 (0.437) ---
Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.


| Retriever | hit@1 | hit@3 |
|---|---|---|
| TF-IDF | 0.67 (4/6) | 0.67 (4/6) |
| Dense (all-MiniLM-L6-v2) | **0.83 (5/6)** | 0.83 (5/6) |

**hit@1 improves from 4/6 to 5/6.** The flooding query is now fixed: the dense model ranks chunk 9 ("riverine inundation… most exposed") first, although it shares no word with "flooding exposure of our plants". This is the vocabulary-mismatch failure that TF-IDF could not handle, and embeddings solve it because they compare meaning, not words.

**The Scope 1 query still fails**, and the relevant chunk 3 is not even in the top 3. The dense model now retrieves **chunk 5 (Targets)**: "reduce Scope 1 and 2 emissions by 35% by 2030". Its error changed: TF-IDF picked the definitions chunk (word overlap), while the dense model picks a chunk that is *about the same topic* (Scope 1 emissions) but gives a **future target, not the reported 2024 value**. Embeddings capture topic well but not finer distinctions like reported vs. target, past vs. future, or a value vs. a definition; and this small general-purpose model does not strongly link "Scope 1" to "direct emissions".

This miss is dangerous: given chunk 5, an LLM could plausibly answer with the target or "not reported" and look perfectly confident, the upstream failure described in Q10 (d). Possible fixes: a **hybrid retriever** (keywords + embeddings), a **re-ranker** that reads query and chunk together, a domain-tuned embedding model, or rewriting the query with synonyms ("Scope 1 / direct emissions, reported value, 2024"). In any case, retrieval quality must be measured separately, as hit@k does here.

---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

**Recommendation: adopt, but only with human checks.** On 20 test companies the model got 87% of values right; the true rate is *very likely* between 78% and 93%. Its main weakness is misreading unusual units ("10,000 tonnes", US short tons) and inventing figures a report does not give. Ask the model five times and have an analyst check every value where the answers disagree (about one in five): total cost EUR 408 per 20 companies, versus EUR 900 with no checks. **Limitation:** we tested only 20 short excerpts, not full reports.